<h1 style="color:DodgerBlue">Индивидуальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Базовый менеджер задач</h2>

----

### Вариант 23


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Task в C#, который будет представлять задачи внутри
проекта. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
public interface IReportable { void GenerateReport(); }
public interface IPrioritizable { void ChangePriority(string newPriority); }
public interface IAuditable { void LogAction(string action); }
public interface IEstimateable
{
    int GetEstimatedHours();
    void SetEstimatedHours(int hours);
}

public delegate void TaskEventHandler(TaskBase task, string message);

public abstract class TaskBase : IReportable, IPrioritizable, IAuditable
{
    protected int taskId;
    protected string taskName;
    protected string priority;
    protected string description;
    protected string createdBy;

    public int TaskId { get => taskId; set => taskId = value; }
    public string TaskName { get => taskName; set => taskName = value; }
    public string Priority { get => priority; set => priority = value; }
    public string Description { get => description; set => description = value; }
    public string CreatedBy { get => createdBy; set => createdBy = value; }

    public bool IsCompleted { get; protected set; }
    public string AssignedTo { get; protected set; }
    public DateTime CreatedDate { get; set; }
    public DateTime LastModified { get; protected set; }
    public string Status { get; set; }
    public int Version { get; protected set; }

    public event TaskEventHandler TaskCompleted;
    public event TaskEventHandler PriorityChanged;

    protected TaskBase()
    {
        TaskId = 0;
        TaskName = "Безымянная задача";
        Priority = "Средний";
        Description = "Описание отсутствует";
        CreatedBy = "Система";
        CreatedDate = DateTime.Now;
        LastModified = DateTime.Now;
        IsCompleted = false;
        AssignedTo = "Не назначен";
        Status = "Активна";
        Version = 1;
    }

    protected TaskBase(int taskId, string taskName, string priority) : this()
    {
        TaskId = taskId;
        TaskName = taskName;
        Priority = priority;
    }

    protected TaskBase(int taskId, string taskName, string priority, string description, string createdBy)
        : this(taskId, taskName, priority)
    {
        Description = description;
        CreatedBy = createdBy;
    }

    public virtual void MarkAsComplete()
    {
        IsCompleted = true;
        Status = "Завершена";
        LastModified = DateTime.Now;
        Version++;
        ((IAuditable)this).LogAction($"Задача '{TaskName}' завершена");
        TaskCompleted?.Invoke(this, $"Задача '{TaskName}' завершена");
    }

    public virtual string GetTaskDetails()
    {
        return $"ID: {TaskId}, Название: {TaskName}, Приоритет: {Priority}, Статус: {Status}, Назначена: {AssignedTo}, Автор: {CreatedBy}, Версия: {Version}";
    }

    public virtual void ReassignTo(string newAssignee)
    {
        AssignedTo = newAssignee;
        LastModified = DateTime.Now;
        Version++;
        ((IAuditable)this).LogAction($"Задача переназначена на {newAssignee}");
    }

    void IReportable.GenerateReport() => GenerateReportCore();

    protected virtual void GenerateReportCore()
    {
        Console.WriteLine($"Отчёт по задаче '{TaskName}':");
        Console.WriteLine($"  Создана: {CreatedDate:yyyy-MM-dd}");
        Console.WriteLine($"  Последнее изменение: {LastModified:yyyy-MM-dd}");
        Console.WriteLine($"  Статус: {Status}");
        Console.WriteLine($"  Версия: {Version}");
    }

    void IPrioritizable.ChangePriority(string newPriority)
    {
        string oldPriority = Priority;
        Priority = newPriority;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Приоритет задачи '{TaskName}' изменён с '{oldPriority}' на '{newPriority}'.");
        PriorityChanged?.Invoke(this, $"Приоритет изменён с {oldPriority} на {newPriority}");
    }

    void IAuditable.LogAction(string action)
    {
        Console.WriteLine($"[{DateTime.Now:yyyy-MM-dd HH:mm}] Задача '{TaskName}': {action}");
    }

    public void UpdateDescription(string newDescription)
    {
        Description = newDescription;
        LastModified = DateTime.Now;
        Version++;
    }

    public void DisplayDescription()
    {
        Console.WriteLine($"Описание задачи '{TaskName}': {Description}");
    }

    public void MarkAsComplete(bool silent)
    {
        IsCompleted = true;
        Status = "Завершена";
        LastModified = DateTime.Now;
        Version++;
        if (!silent)
            ((IAuditable)this).LogAction($"Задача '{TaskName}' завершена");
    }

    public void MarkAsComplete(string comment)
    {
        IsCompleted = true;
        Status = "Завершена";
        LastModified = DateTime.Now;
        Version++;
        ((IAuditable)this).LogAction($"Задача '{TaskName}' завершена с комментарием: {comment}");
    }
}

public class DelegateTask : TaskBase, IEstimateable
{
    public DateTime DueDate { get; set; }
    public string DelegatedBy { get; set; }
    public bool IsUrgent { get; set; }
    public string DelegationChannel { get; set; }
    public int DelegationLevel { get; set; }

    private int estimatedHours;

    public DelegateTask() : base()
    {
        DueDate = DateTime.Now.AddDays(7);
        DelegatedBy = "Не указан";
        IsUrgent = false;
        estimatedHours = 1;
        DelegationChannel = "Внутренний";
        DelegationLevel = 1;
    }

    public DelegateTask(int taskId, string taskName, string priority, DateTime dueDate)
        : base(taskId, taskName, priority)
    {
        DueDate = dueDate;
        DelegatedBy = "Не указан";
        IsUrgent = false;
        estimatedHours = 1;
        DelegationChannel = "Внутренний";
        DelegationLevel = 1;
    }

    public DelegateTask(int taskId, string taskName, string priority, DateTime dueDate,
        string delegatedBy, bool isUrgent, int estimatedHours)
        : base(taskId, taskName, priority)
    {
        DueDate = dueDate;
        DelegatedBy = delegatedBy;
        IsUrgent = isUrgent;
        this.estimatedHours = estimatedHours;
        DelegationChannel = "Внутренний";
        DelegationLevel = 1;
    }

    public override void MarkAsComplete()
    {
        base.MarkAsComplete();
        Console.WriteLine($"Делегатская задача '{TaskName}' выполнена. Срок: {DueDate:yyyy-MM-dd}");
    }

    public override string GetTaskDetails()
    {
        return base.GetTaskDetails() +
               $", Срок: {DueDate:yyyy-MM-dd}, Делегирована: {DelegatedBy}, Срочная: {IsUrgent}, Оценка: {estimatedHours} ч, Канал: {DelegationChannel}, Уровень: {DelegationLevel}";
    }

    protected override void GenerateReportCore()
    {
        base.GenerateReportCore();
        Console.WriteLine($"  Срок выполнения: {DueDate:yyyy-MM-dd}");
        Console.WriteLine($"  Делегирована: {DelegatedBy}");
        Console.WriteLine($"  Срочность: {(IsUrgent ? "Срочная" : "Обычная")}");
        Console.WriteLine($"  Оценка времени: {estimatedHours} ч");
        Console.WriteLine($"  Канал делегирования: {DelegationChannel}");
        Console.WriteLine($"  Уровень делегирования: {DelegationLevel}");
    }

    public void CheckDeadline()
    {
        if (DateTime.Now > DueDate)
            Console.WriteLine($"Задача '{TaskName}' просрочена!");
        else
            Console.WriteLine($"Задача '{TaskName}' в срок. Дней осталось: {(DueDate - DateTime.Now).Days}");
    }

    public void ExtendDeadline(int days)
    {
        DueDate = DueDate.AddDays(days);
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Срок задачи '{TaskName}' продлён на {days} дней. Новый срок: {DueDate:yyyy-MM-dd}");
    }

    public void ExtendDeadline(TimeSpan period)
    {
        DueDate = DueDate.Add(period);
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Срок задачи '{TaskName}' продлён на {period.Days} дней. Новый срок: {DueDate:yyyy-MM-dd}");
    }

    public void Escalate()
    {
        DelegationLevel++;
        IsUrgent = true;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Задача '{TaskName}' эскалирована. Уровень: {DelegationLevel}");
    }

    int IEstimateable.GetEstimatedHours() => estimatedHours;

    void IEstimateable.SetEstimatedHours(int hours)
    {
        estimatedHours = hours;
        LastModified = DateTime.Now;
        Version++;
    }
}

public class TeamTask : TaskBase
{
    public string TeamName { get; set; }
    public int TeamSize { get; set; }
    public string TeamLead { get; set; }
    public string ProjectCode { get; set; }
    public string CommunicationTool { get; set; }
    public bool IsCrossFunctional { get; set; }

    public TeamTask() : base()
    {
        TeamName = "Общая команда";
        TeamSize = 1;
        TeamLead = "Не указан";
        ProjectCode = "N/A";
        CommunicationTool = "Email";
        IsCrossFunctional = false;
    }

    public TeamTask(int taskId, string taskName, string priority, string teamName)
        : base(taskId, taskName, priority)
    {
        TeamName = teamName;
        TeamSize = 1;
        TeamLead = "Не указан";
        ProjectCode = "N/A";
        CommunicationTool = "Email";
        IsCrossFunctional = false;
    }

    public TeamTask(int taskId, string taskName, string priority, string teamName,
        int teamSize, string teamLead, string projectCode)
        : base(taskId, taskName, priority)
    {
        TeamName = teamName;
        TeamSize = teamSize;
        TeamLead = teamLead;
        ProjectCode = projectCode;
        CommunicationTool = "Email";
        IsCrossFunctional = false;
    }

    public override void ReassignTo(string newAssignee)
    {
        string oldAssignee = AssignedTo;
        base.ReassignTo(newAssignee);
        Console.WriteLine($"Задача '{TaskName}' переназначена с {oldAssignee} на {newAssignee} в команде '{TeamName}'.");
    }

    public override string GetTaskDetails()
    {
        return base.GetTaskDetails() +
               $", Команда: {TeamName}, Размер команды: {TeamSize}, Лидер: {TeamLead}, Проект: {ProjectCode}, Инструмент: {CommunicationTool}, Кросс-функциональная: {IsCrossFunctional}";
    }

    protected override void GenerateReportCore()
    {
        base.GenerateReportCore();
        Console.WriteLine($"  Команда: {TeamName}");
        Console.WriteLine($"  Размер команды: {TeamSize}");
        Console.WriteLine($"  Лидер: {TeamLead}");
        Console.WriteLine($"  Код проекта: {ProjectCode}");
        Console.WriteLine($"  Инструмент коммуникации: {CommunicationTool}");
        Console.WriteLine($"  Кросс-функциональная: {(IsCrossFunctional ? "Да" : "Нет")}");
    }

    public void AssignTeamMember(string member)
    {
        TeamSize++;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"{member} назначен в команду '{TeamName}'. Новый размер команды: {TeamSize}");
    }

    public void AssignTeamMember(string member, string role)
    {
        TeamSize++;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"{member} назначен в команду '{TeamName}' на роль '{role}'. Новый размер команды: {TeamSize}");
    }

    public void RemoveTeamMember(string member)
    {
        if (TeamSize > 1)
        {
            TeamSize--;
            LastModified = DateTime.Now;
            Version++;
            Console.WriteLine($"{member} удалён из команды '{TeamName}'. Новый размер команды: {TeamSize}");
        }
        else
        {
            Console.WriteLine($"Нельзя удалить последнего члена команды '{TeamName}'.");
        }
    }

    public void ChangeTeamLead(string newLead)
    {
        string oldLead = TeamLead;
        TeamLead = newLead;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Лидер команды '{TeamName}' изменён с {oldLead} на {newLead}.");
    }

    public void MarkAsCrossFunctional()
    {
        IsCrossFunctional = true;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Команда '{TeamName}' помечена как кросс-функциональная.");
    }
}

public class ResearchTask : TaskBase, IEstimateable
{
    public string DataSource { get; set; }
    public string ResearchMethod { get; set; }
    public int SampleSize { get; set; }
    public bool IsPeerReviewed { get; set; }
    public string ResearchArea { get; set; }
    public double ConfidenceLevel { get; set; }

    private int plannedHours;

    public ResearchTask() : base()
    {
        DataSource = "Открытые источники";
        ResearchMethod = "Наблюдение";
        SampleSize = 0;
        IsPeerReviewed = false;
        ResearchArea = "Общая";
        ConfidenceLevel = 0.95;
        plannedHours = 0;
    }

    public ResearchTask(int taskId, string taskName, string priority, string dataSource)
        : base(taskId, taskName, priority)
    {
        DataSource = dataSource;
        ResearchMethod = "Наблюдение";
        SampleSize = 0;
        IsPeerReviewed = false;
        ResearchArea = "Общая";
        ConfidenceLevel = 0.95;
        plannedHours = 0;
    }

    public ResearchTask(int taskId, string taskName, string priority, string dataSource,
        string researchMethod, int sampleSize, bool isPeerReviewed)
        : base(taskId, taskName, priority)
    {
        DataSource = dataSource;
        ResearchMethod = researchMethod;
        SampleSize = sampleSize;
        IsPeerReviewed = isPeerReviewed;
        ResearchArea = "Общая";
        ConfidenceLevel = 0.95;
        plannedHours = 0;
    }

    public override string GetTaskDetails()
    {
        return base.GetTaskDetails() +
               $", Источник данных: {DataSource}, Метод: {ResearchMethod}, Выборка: {SampleSize}, Рецензировано: {IsPeerReviewed}, Плановые часы: {plannedHours}, Область: {ResearchArea}, Доверительный уровень: {ConfidenceLevel}";
    }

    protected override void GenerateReportCore()
    {
        base.GenerateReportCore();
        Console.WriteLine($"  Источник данных: {DataSource}");
        Console.WriteLine($"  Метод исследования: {ResearchMethod}");
        Console.WriteLine($"  Размер выборки: {SampleSize}");
        Console.WriteLine($"  Рецензировано: {(IsPeerReviewed ? "Да" : "Нет")}");
        Console.WriteLine($"  Плановые часы: {plannedHours}");
        Console.WriteLine($"  Область исследования: {ResearchArea}");
        Console.WriteLine($"  Доверительный уровень: {ConfidenceLevel}");
    }

    public override void MarkAsComplete()
    {
        base.MarkAsComplete();
        Console.WriteLine($"Исследовательская задача '{TaskName}' выполнена.");
    }

    public void AnalyzeData(TaskBase relatedTask)
    {
        Console.WriteLine($"Анализ данных из источника '{DataSource}' по задаче '{TaskName}'.");
        Console.WriteLine($"Связанная задача: '{relatedTask.TaskName}'.");
        ((IAuditable)this).LogAction($"Проведён анализ данных по задаче '{relatedTask.TaskName}'");
    }

    public void AddDataSource(string source)
    {
        DataSource += ", " + source;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Добавлен источник данных '{source}' к задаче '{TaskName}'.");
    }

    public void AddDataSource(string source, string description)
    {
        DataSource += ", " + source + " (" + description + ")";
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Добавлен источник данных '{source}' ({description}) к задаче '{TaskName}'.");
    }

    public void MarkAsPeerReviewed()
    {
        IsPeerReviewed = true;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Задача '{TaskName}' прошла рецензирование.");
    }

    public void SetConfidenceLevel(double level)
    {
        ConfidenceLevel = level;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Доверительный уровень задачи '{TaskName}' установлен на {level}.");
    }

    int IEstimateable.GetEstimatedHours() => plannedHours;

    void IEstimateable.SetEstimatedHours(int hours)
    {
        plannedHours = hours;
        LastModified = DateTime.Now;
        Version++;
    }
}

public class ScientificResearchTask : ResearchTask
{
    public string ScientificField { get; set; }
    public string Hypothesis { get; set; }
    public bool IsPublished { get; set; }
    public string JournalName { get; set; }
    public int CitationCount { get; set; }

    public ScientificResearchTask() : base()
    {
        ScientificField = "Общая наука";
        Hypothesis = "Гипотеза не сформулирована";
        IsPublished = false;
        JournalName = "Не выбран";
        CitationCount = 0;
    }

    public ScientificResearchTask(int taskId, string taskName, string priority, string dataSource,
        string researchMethod, int sampleSize, bool isPeerReviewed, string scientificField, string hypothesis)
        : base(taskId, taskName, priority, dataSource, researchMethod, sampleSize, isPeerReviewed)
    {
        ScientificField = scientificField;
        Hypothesis = hypothesis;
        IsPublished = false;
        JournalName = "Не выбран";
        CitationCount = 0;
    }

    public override string GetTaskDetails()
    {
        return base.GetTaskDetails() +
               $", Область науки: {ScientificField}, Гипотеза: {Hypothesis}, Опубликовано: {IsPublished}, Журнал: {JournalName}, Цитирований: {CitationCount}";
    }

    protected override void GenerateReportCore()
    {
        base.GenerateReportCore();
        Console.WriteLine($"  Область науки: {ScientificField}");
        Console.WriteLine($"  Гипотеза: {Hypothesis}");
        Console.WriteLine($"  Опубликовано: {(IsPublished ? "Да" : "Нет")}");
        Console.WriteLine($"  Журнал: {JournalName}");
        Console.WriteLine($"  Цитирований: {CitationCount}");
    }

    public void Publish()
    {
        IsPublished = true;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Научная работа '{TaskName}' опубликована в области '{ScientificField}'.");
    }

    public void FormulateHypothesis(string hypothesis)
    {
        Hypothesis = hypothesis;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Гипотеза задачи '{TaskName}' сформулирована: {hypothesis}");
    }

    public void AddCitation()
    {
        CitationCount++;
        LastModified = DateTime.Now;
        Version++;
        Console.WriteLine($"Задача '{TaskName}' получила новое цитирование. Всего: {CitationCount}");
    }
}

public class TaskCollection<T> where T : TaskBase
{
    private List<T> _tasks = new List<T>();

    public void Add(T task) => _tasks.Add(task);
    public void Remove(T task) => _tasks.Remove(task);

    public void DisplayAll()
    {
        foreach (var task in _tasks)
        {
            Console.WriteLine(task.GetTaskDetails());
            Console.WriteLine();
        }
    }

    public void CompleteAll()
    {
        foreach (var task in _tasks)
            task.MarkAsComplete();
    }

    public void GenerateReports()
    {
        foreach (var task in _tasks)
        {
            ((IReportable)task).GenerateReport();
            Console.WriteLine();
        }
    }

    public IEnumerable<T> Find(Func<T, bool> predicate) => _tasks.Where(predicate);

    public int Count => _tasks.Count;
}

public class TaskDependencyService
{
    private readonly IReportable _reportable;
    private readonly IPrioritizable _prioritizable;
    private readonly IAuditable _auditable;

    public TaskDependencyService(TaskBase task)
    {
        _reportable = task;
        _prioritizable = task;
        _auditable = task;
    }

    public void ProcessTask(string newPriority, string action)
    {
        _reportable.GenerateReport();
        _prioritizable.ChangePriority(newPriority);
        _auditable.LogAction(action);
    }
}

var delegateTask = new DelegateTask(1, "Документация", "Высокий", DateTime.Now.AddDays(5), "Руководитель", false, 8);
var teamTask = new TeamTask(2, "Аутентификация", "Критический", "Бэкенд", 5, "Игорь", "PRJ-2024");
var researchTask = new ResearchTask(3, "Анализ данных", "Средний", "Google Analytics", "Статистический анализ", 1000, false);
var scienceTask = new ScientificResearchTask(4, "Квантовое исследование", "Высокий", "CERN", "Эксперимент", 5000, true, "Физика", "Гипотеза о поведении частиц");

delegateTask.TaskCompleted += (task, msg) => Console.WriteLine($"[СОБЫТИЕ] {msg}");
delegateTask.PriorityChanged += (task, msg) => Console.WriteLine($"[СОБЫТИЕ] {msg}");

Console.WriteLine(delegateTask.GetTaskDetails());
Console.WriteLine(teamTask.GetTaskDetails());
Console.WriteLine(researchTask.GetTaskDetails());
Console.WriteLine(scienceTask.GetTaskDetails());

Console.WriteLine();

delegateTask.ReassignTo("Анна");
teamTask.ReassignTo("Игорь");
researchTask.ReassignTo("Дмитрий");
scienceTask.ReassignTo("Елена");

delegateTask.CheckDeadline();
delegateTask.ExtendDeadline(3);
delegateTask.ExtendDeadline(TimeSpan.FromDays(2));
delegateTask.Escalate();
teamTask.AssignTeamMember("Петр");
teamTask.AssignTeamMember("Ольга", "Тестировщик");
teamTask.RemoveTeamMember("Петр");
teamTask.ChangeTeamLead("Мария");
teamTask.MarkAsCrossFunctional();
researchTask.AddDataSource("Опросы");
researchTask.AddDataSource("Интервью", "Глубинное интервью с пользователями");
researchTask.MarkAsPeerReviewed();
researchTask.SetConfidenceLevel(0.99);
scienceTask.FormulateHypothesis("Новая гипотеза о поведении частиц");
scienceTask.Publish();
scienceTask.AddCitation();

Console.WriteLine();

var service = new TaskDependencyService(delegateTask);
service.ProcessTask("Критический", "Задача обработана через сервис зависимостей");

Console.WriteLine();

((IReportable)delegateTask).GenerateReport();
Console.WriteLine();
((IReportable)teamTask).GenerateReport();
Console.WriteLine();
((IReportable)researchTask).GenerateReport();
Console.WriteLine();
((IReportable)scienceTask).GenerateReport();

Console.WriteLine();

var allTasks = new TaskCollection<TaskBase>();
allTasks.Add(delegateTask);
allTasks.Add(teamTask);
allTasks.Add(researchTask);
allTasks.Add(scienceTask);
allTasks.DisplayAll();

var urgentTasks = allTasks.Find(t => t.Priority == "Критический");
Console.WriteLine("Критические задачи:");
foreach (var t in urgentTasks)
    Console.WriteLine($"  - {t.TaskName}");

Console.WriteLine($"Общее количество задач: {allTasks.Count}");

ID: 1, Название: Документация, Приоритет: Высокий, Статус: Активна, Назначена: Не назначен, Автор: Система, Версия: 1, Срок: 2026-11-23, Делегирована: Руководитель, Срочная: False, Оценка: 8 ч, Канал: Внутренний, Уровень: 1
ID: 2, Название: Аутентификация, Приоритет: Критический, Статус: Активна, Назначена: Не назначен, Автор: Система, Версия: 1, Команда: Бэкенд, Размер команды: 5, Лидер: Игорь, Проект: PRJ-2024, Инструмент: Email, Кросс-функциональная: False
ID: 3, Название: Анализ данных, Приоритет: Средний, Статус: Активна, Назначена: Не назначен, Автор: Система, Версия: 1, Источник данных: Google Analytics, Метод: Статистический анализ, Выборка: 1000, Рецензировано: False, Плановые часы: 0, Область: Общая, Доверительный уровень: 0,95
ID: 4, Название: Квантовое исследование, Приоритет: Высокий, Статус: Активна, Назначена: Не назначен, Автор: Система, Версия: 1, Источник данных: CERN, Метод: Эксперимент, Выборка: 5000, Рецензировано: True, Плановые часы: 0, Область: Общая, Доверитель